# Machine Learning Task 2 - Restaurant Recommendation System

This notebook loads a restaurant dataset, cleans missing values, prepares features, and recommends similar restaurants using K-Nearest Neighbors.

## Step 1: Install Required Libraries

Run this cell once in VS Code. If the libraries are already installed, it will simply confirm that they are available.

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn matplotlib seaborn ipykernel


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.neighbors import NearestNeighbors

## Step 3: Load Dataset

Keep `Dataset .csv` in the same folder as this notebook. The filename has a space before `.csv`, so it must match exactly.

In [3]:
csv_path = Path("Dataset .csv")

if not csv_path.exists():
    csv_path = Path.home() / "Downloads" / "Dataset .csv"

if not csv_path.exists():
    raise FileNotFoundError("Dataset .csv not found. Please keep Dataset .csv in the same folder as this notebook.")

dataset = pd.read_csv(csv_path)

print("Dataset loaded successfully from:", csv_path)
print("Dataset shape:", dataset.shape)

Dataset loaded successfully from: C:\Users\ajay2\Downloads\Dataset .csv
Dataset shape: (9551, 21)


## Step 4: Basic Data Analysis

In [4]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [5]:
dataset.head(15)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229
5,18189371,Din Tai Fung,162,Mandaluyong City,"Ground Floor, Mega Fashion Hall, SM Megamall, ...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056314,14.583764,Chinese,...,Botswana Pula(P),No,No,No,No,3,4.4,Green,Very Good,336
6,6300781,Buffet 101,162,Pasay City,"Building K, SM By The Bay, Sunset Boulevard, M...","SM by the Bay, Mall of Asia Complex, Pasay City","SM by the Bay, Mall of Asia Complex, Pasay Cit...",120.979667,14.531333,"Asian, European",...,Botswana Pula(P),Yes,No,No,No,4,4.0,Green,Very Good,520
7,6301290,Vikings,162,Pasay City,"Building B, By The Bay, Seaside Boulevard, Mal...","SM by the Bay, Mall of Asia Complex, Pasay City","SM by the Bay, Mall of Asia Complex, Pasay Cit...",120.979333,14.540000,"Seafood, Filipino, Asian, European",...,Botswana Pula(P),Yes,No,No,No,4,4.2,Green,Very Good,677
8,6300010,Spiral - Sofitel Philippine Plaza Manila,162,Pasay City,"Plaza Level, Sofitel Philippine Plaza Manila, ...","Sofitel Philippine Plaza Manila, Pasay City","Sofitel Philippine Plaza Manila, Pasay City, P...",120.980090,14.552990,"European, Asian, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.9,Dark Green,Excellent,621
9,6314987,Locavore,162,Pasig City,"Brixton Technology Center, 10 Brixton Street, ...",Kapitolyo,"Kapitolyo, Pasig City",121.056532,14.572041,Filipino,...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,532


In [6]:
print("Number of rows and columns:", dataset.shape)
print("\nColumn names:")
print(dataset.columns)

Number of rows and columns: (9551, 21)

Column names:
Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes'],
      dtype='object')


## Step 5: Check Missing Values

In [7]:
missing_values = pd.DataFrame({
    "Missing Values": dataset.isnull().sum()
})

missing_values

,Missing Values
Restaurant ID,0
Restaurant Name,0
Country Code,0
City,0
Address,0
Locality,0
Locality Verbose,0
Longitude,0
Latitude,0
Cuisines,9


## Step 6: Handle Missing Values

In [8]:
dataset["Cuisines"] = dataset["Cuisines"].fillna(dataset["Cuisines"].mode()[0])
dataset["City"] = dataset["City"].fillna(dataset["City"].mode()[0])

dataset["Average Cost for two"] = dataset["Average Cost for two"].fillna(
    dataset["Average Cost for two"].median()
)

dataset["Aggregate rating"] = dataset["Aggregate rating"].fillna(
    dataset["Aggregate rating"].median()
)

print("Missing values after cleaning:")
print(dataset.isnull().sum())

Missing values after cleaning:
Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                0
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64


## Step 7: Label Encoding

Machine learning algorithms work with numbers, so city names are converted into numerical labels.

In [9]:
city_encoder = LabelEncoder()
dataset["City_encoded"] = city_encoder.fit_transform(dataset["City"])

dataset[["City", "City_encoded"]].head(10)

,City,City_encoded
0,Makati City,73
1,Makati City,73
2,Mandaluyong City,75
3,Mandaluyong City,75
4,Mandaluyong City,75
5,Mandaluyong City,75
6,Pasay City,94
7,Pasay City,94
8,Pasay City,94
9,Pasig City,95


In [10]:
print("Encoded city values:")
print(dataset["City_encoded"].unique())

print("\nOriginal city names:")
print(dataset["City"].unique())

Encoded city values:
[ 73  75  94  95 107 112 114 122 123  21 111 121   3   7   8  10  12  15
  20  23  25  27  28  31  32  33  34  36  37  40  41  44  45  46  47  52
  53  57  64  65  66  68  71  72  77  78  79  80  82  83  90  91  92  97
  98  99 100 101 102 104 110 115 118 119 124 126 127 129 131 132 133 135
 136 138 139   0  39 117   1   2   4   5  11  14  16  17  24  26  29  35
  43  48  49  50  51  54  55  58  61  62  63  69  70  76  81  84  85  86
  87  88  89  93  96 105 106 108 116 120 128 130 134  13  19  59 125   9
 137  18  42  67  74  38  22  56  60 103 109 113  30   6 140]

Original city names:
['Makati City' 'Mandaluyong City' 'Pasay City' 'Pasig City' 'Quezon City'
 'San Juan City' 'Santa Rosa' 'Tagaytay City' 'Taguig City' 'Bras�_lia'
 'Rio de Janeiro' 'S��o Paulo' 'Albany' 'Armidale' 'Athens' 'Augusta'
 'Balingup' 'Beechworth' 'Boise' 'Cedar Rapids/Iowa City' 'Chatham-Kent'
 'Clatskanie' 'Cochrane' 'Columbus' 'Consort' 'Dalton' 'Davenport'
 'Des Moines' 'Dicky Beach' 

## Step 8: Feature Scaling

Scaling converts numerical values into a similar range, which helps distance-based algorithms like KNN.

In [11]:
rating_scaler = MinMaxScaler()
cost_scaler = MinMaxScaler()
city_scaler = MinMaxScaler()

dataset["Average_Rating_scaled"] = rating_scaler.fit_transform(dataset[["Aggregate rating"]])
dataset["Average_Cost_scaled"] = cost_scaler.fit_transform(dataset[["Average Cost for two"]])
dataset["City_encoded_scaled"] = city_scaler.fit_transform(dataset[["City_encoded"]])

feature_columns = [
    "City_encoded_scaled",
    "Average_Rating_scaled",
    "Average_Cost_scaled"
]

X = dataset[feature_columns].values

print("Feature matrix created successfully.")
print("Feature matrix shape:", X.shape)

Feature matrix created successfully.
Feature matrix shape: (9551, 3)


## Step 9: Build KNN Model

KNN finds restaurants that are closest to the selected restaurant using Euclidean distance.

In [12]:
knn = NearestNeighbors(n_neighbors=6, metric="euclidean")
knn.fit(X)

print("KNN model trained successfully.")

KNN model trained successfully.


## Step 10: Get Restaurant Recommendations

In [13]:
restaurant_idx = 15

if restaurant_idx >= len(dataset):
    raise IndexError("restaurant_idx is outside the dataset range.")

distances, indices = knn.kneighbors(X[restaurant_idx].reshape(1, -1))

print("Selected Restaurant:")
print(dataset.loc[restaurant_idx, [
    "Restaurant Name",
    "City",
    "Address",
    "Cuisines",
    "Average Cost for two",
    "Aggregate rating"
]])

print("\nRecommended restaurant indices:")
print(indices)

print("\nDistances to recommended restaurants:")
print(distances)

Selected Restaurant:
Restaurant Name                                             Cafe Arabelle
City                                                           Santa Rosa
Address                 Ayala Mall, Solenad, Nuvali, Santa Rosa - Taga...
Cuisines                                Cafe, American, Italian, Filipino
Average Cost for two                                                  800
Aggregate rating                                                      3.6
Name: 15, dtype: object

Recommended restaurant indices:
[[ 15 441 466 481 477 479]]

Distances to recommended restaurants:
[[0.         0.02164375 0.03512438 0.03572742 0.03572794 0.03572794]]


In [14]:
# The first result is the selected restaurant itself, so we skip it using [1:].
recommended_restaurants = dataset.iloc[indices[0][1:]]

recommended_restaurants[[
    "Restaurant Name",
    "City",
    "Address",
    "Cuisines",
    "Average Cost for two",
    "Aggregate rating",
    "Votes"
]]

,Restaurant Name,City,Address,Cuisines,Average Cost for two,Aggregate rating,Votes
441,Moon River Brewing Company,Savannah,"21 W Bay St, Savannah, GA 31401","American, Bar Food, Sandwich",25,3.7,747
466,Chye Seng Huat Hardware,Singapore,150 Tyrwhitt Road 207563,Cafe,40,3.7,33
481,HuHot Mongolian Grill,Sioux City,"4229 S Lakeport St, Sioux City, IA 51106","Asian, Chinese",25,3.6,94
477,Bob Roe's Pizza,Sioux City,"2320 Transit Ave, Sioux City, IA 51106","American, Pizza, Bar Food",10,3.6,92
479,Famous Dave's,Sioux City,"201 Pierce St, Sioux City, IA 51101",BBQ,10,3.6,76


## Viva Explanation

This project is a restaurant recommendation system. First, the restaurant dataset is loaded using pandas. Then basic data analysis is performed by checking rows, columns, data types, and missing values. Missing values are handled using mode for categorical columns and median for numerical columns. Since machine learning models need numerical input, city names are converted into numbers using label encoding. Rating, cost, and city encoded values are scaled using MinMaxScaler. Finally, the K-Nearest Neighbors algorithm is used to find restaurants similar to a selected restaurant based on city, rating, and average cost.